# Tags on Tensor Networks

In [1]:
using Tangles

`TaggedTensorNetwork` is the interface that allows a `TensorNetwork` to tag its tensors and indices with `Tag`s.

`GenericTensorNetwork` is a concrete type that wraps `SimpleTensorNetwork` and implements `TaggedTensorNetwork`.

In [2]:
tn = GenericTensorNetwork()

GenericTensorNetwork (#tensors=0, #inds=0)

In [3]:
Γa = NamedTensor([1 0; 0 1], [Index(:i), Index(:j)])
Γb = NamedTensor([1 0; 0 1], [Index(:j), Index(:k)])
addtensor!(tn, Γa)
addtensor!(tn, Γb)

SimpleTensorNetwork (#tensors=2, #inds=3)

It associates vertices and edges with `Site` and `Link` tags.

`Site` is a `Tag` abstract type that should be thought as a "physical site" in physical lattice. The most basic concrete subtype is `CartesianSite`, which stores a `Tuple` of integers.

In [4]:
@show CartesianSite(1) CartesianSite(2,3)

CartesianSite(1) = site<1>
CartesianSite(2, 3) = site<2,3>


site<2,3>

In other to facilitate writing, there is the `@site_str` macro that translates to a `CartesianSite`.

In [5]:
site"1"

site<1>

In [6]:
site"1,2"

site<1,2>

In [7]:
typeof(site"1")

CartesianSite{1}

In order to tag a vertex with a `Site`, there is the `setsite!` function.

In [8]:
setsite!(tn, Γa, site"1")

GenericTensorNetwork (#tensors=2, #inds=3)

Note that since `Tensor`s are associated one-to-one with vertices, they can also be use in substitution of vertices; i.e. you can directly tag `Tensor`s.

In [9]:
setsite!(tn, Γb, site"2")

GenericTensorNetwork (#tensors=2, #inds=3)

`sites` returns all the tags associated to `Tensor`s (i.e. all the `Site` tags).

In [10]:
sites(tn)

2-element Vector{Tangles.Site}:
 site<2>
 site<1>

In order to retrieve a vertex or tensor with that tag, there are the `tensor_at` (aka `tensor(; at)`) functions.

In [11]:
tensor_at(tn, site"1")

2×2 NamedTensor(::Tensor(::Matrix{Int64})) with signature --:
 1  0
 0  1

In [12]:
tensor(tn; at=site"1") === Γa

true

On the other hand, there are `Link`s: `Tag` abstract type for `Index`s.

The most fundamental `Link` types are `Plug`, for representing physical input/output open indices, and `Bond`, for representing virtual bonds connecting different sites.

In [13]:
plug"1"

plug<site<1>>

In [14]:
typeof(plug"1")

Plug{CartesianSite{1}}

In [15]:
bond"1-2"

bond<site<1> ⟷ site<2>>

In [16]:
Bond(site"1", site"2")

bond<site<1> ⟷ site<2>>

Just like with `Site`s, `setlink!` can be used for tagging `Index` with a `Link`.

In [17]:
setlink!(tn, Index(:i), plug"1")

GenericTensorNetwork (#tensors=2, #inds=3)

In [18]:
setlink!(tn, Index(:k), plug"2")
setlink!(tn, Index(:j), bond"1-2")

GenericTensorNetwork (#tensors=2, #inds=3)

`links` retrieves all the `Tag`s associated with `Index`s.

In [19]:
links(tn)

3-element Vector{Tangles.Link}:
 plug<site<2>>
 plug<site<1>>
 bond<site<1> ⟷ site<2>>

Finally, `ind_at` (aka `ind(; at)`) return the `Index` linked to the tag.

In [20]:
ind_at(tn, plug"1")

i

## The `TensorNetwork` underneath

`GenericTensorNetwork` implements both `TensorNetwork` (through the delegation to `SimpleTensorNetwork`) and `Taggable`, but a method from one of the interfaces might mutate the `GenericTensorNetwork` in such a way that the information from the other interface might be invalidated. It is the responsability of the implementor type (i.e. `GenericTensorNetwork`) to fix any inconsistencies.

For example, calling `rmtensor!` on `GenericTensorNetwork` will automatically be delegated to the `SimpleTensorNetwork` and the `Tensor` (and `Vertex`) will be removed. If the `Tensor` is associated to a `Site`, removing the `Tensor` and not the `Site` will create a inconsistency.
Due to that, `GenericTensorNetwork` intercepts the `rmtensor!`, removes the tag and finally calls `rmtensor!` on its delegated field (i.e. on the `SimpleTensorNetwork`).
Another example is that `addvertex!` or `rmvertex!` from the `Network` interface on `SimpleTensorNetwork` would break the one-to-one map between vertices and `Tensor`s, and thus `SimpleTensorNetwork` forbids them.

Summarizing, calling mutating functions of interfaces that are delegated should be safe or forbidden (note for implementors).

In [21]:
γa = similar(Γa)
replace!(tn, Γa => γa)

GenericTensorNetwork (#tensors=2, #inds=3)

In [22]:
site_at(tn, Γa)

KeyError: KeyError: key [1 0; 0 1] not found

`site_at(tn, Γa)` fails because `Γa` is no longer in the Tensor Network, but the tag remains associated with the new `Tensor` it has been replaced with.

In [23]:
site_at(tn, γa)

site<1>

## Creating a new `Tag` type

Creating a new `Tag` type is extremely easy. You just keep in mind that you might want to overload some methods related to somo other `Tag` if it needs to interact with or act like it.

In [24]:
struct VidalLambda{B} <: Tangles.Site
    bond::B
end

Tangles.bond(x::VidalLambda) = x.bond

In [25]:
Λ = NamedTensor([1, 1], [Index(:j)])

addtensor!(tn, Λ)
setsite!(tn, Λ, VidalLambda(bond"1-2"))

GenericTensorNetwork (#tensors=3, #inds=3)

In [26]:
tensor(tn; at=VidalLambda(bond"1-2"))

2-element NamedTensor(::Tensor(::Vector{Int64})) with signature -:
 1
 1

Alternatively, `Base.getindex` and `Base.setindex!` can be used for querying and assigning tensors and links. 

In [27]:
tn = GenericTensorNetwork()
tn[VidalLambda(bond"1-2")] = Λ

2-element NamedTensor(::Tensor(::Vector{Int64})) with signature -:
 1
 1

In [28]:
tn[VidalLambda(bond"1-2")]

2-element NamedTensor(::Tensor(::Vector{Int64})) with signature -:
 1
 1